# Évaluation du routage d'intentions

Ce notebook évalue la capacité du chatbot AMENet à router correctement différents types de messages utilisateur.

Objectifs :

- vérifier la détection des intentions principales ;
- vérifier que les actions sensibles demandent une confirmation explicite ;
- vérifier que les questions documentaires sont routées vers le RAG ;
- vérifier que les questions hors périmètre sont refusées.

Le notebook utilise le backend Python directement, sans passer par l'API HTTP FastAPI.

## 1. Initialisation

On ajoute la racine du projet au `PYTHONPATH`, puis on désactive volontairement Ollama pour cette évaluation.

L'objectif de ce notebook est d'évaluer le routage d'intentions, pas la qualité de génération du LLM local.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# Si le notebook est lancé depuis le dossier notebooks/, on remonte d'un niveau.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

# Pour rendre l'évaluation rapide et déterministe.
os.environ["OLLAMA_ENABLED"] = "false"

PROJECT_ROOT

## 2. Imports et création du service de chat

In [ ]:
import pandas as pd
from types import SimpleNamespace

from backend.app.services.chat_service import ChatService


class FakeRagService:
    def search_documents(self, query: str, top_k: int = 3):
        return [
            SimpleNamespace(
                id="fake_rag_chunk",
                text=(
                    "Dans le prototype, les actions sensibles comme l'opposition sur carte "
                    "nécessitent une confirmation explicite avant d'être enregistrées dans "
                    "l'environnement de simulation."
                ),
                title="Source RAG simulée pour évaluation du routage",
                source_type="test",
                source_file="notebook_fake_source",
                source_image=None,
                page=None,
                domain="evaluation",
                tags=["test"],
                score=0.99,
                metadata={},
            )
        ]


class FakeOllamaAnswerer:
    def generate_answer(self, query: str, results: list):
        return None


def create_chat_service() -> ChatService:
    return ChatService(
        rag_service=FakeRagService(),
        ollama_answerer=FakeOllamaAnswerer(),
    )


print("Fonction create_chat_service() initialisée avec un faux RAG léger.")


## 3. Jeu de cas de test

Chaque cas contient :

- un message utilisateur ;
- l'intention attendue ;
- l'information indiquant si une confirmation est attendue ;
- une description du comportement testé.

In [ ]:
test_cases = [
    {
        "message": "Quel est mon solde ?",
        "expected_intent": "get_balance",
        "expected_requires_confirmation": False,
        "category": "consultation",
        "description": "Consultation du solde fictif",
    },
    {
        "message": "Affiche mes dernières opérations",
        "expected_intent": "get_transactions",
        "expected_requires_confirmation": False,
        "category": "consultation",
        "description": "Consultation des mouvements fictifs",
    },
    {
        "message": "Je veux faire un virement de 500 DT",
        "expected_intent": "prepare_transfer",
        "expected_requires_confirmation": True,
        "category": "action_sensible",
        "description": "Préparation d'un virement simulé",
    },
    {
        "message": "Je veux bloquer ma carte qui termine par 4582",
        "expected_intent": "block_card",
        "expected_requires_confirmation": True,
        "category": "action_sensible",
        "description": "Opposition sur carte simulée",
    },
    {
        "message": "Je veux bloquer une carte",
        "expected_intent": "block_card",
        "expected_requires_confirmation": True,
        "category": "action_sensible",
        "description": "Détection de secours d'une action de blocage carte",
    },
    {
        "message": "Je veux commander un chéquier",
        "expected_intent": "request_checkbook",
        "expected_requires_confirmation": True,
        "category": "action_sensible",
        "description": "Demande de chéquier simulée",
    },
    {
        "message": "Je veux demander un relevé",
        "expected_intent": "request_document",
        "expected_requires_confirmation": True,
        "category": "action_sensible",
        "description": "Demande de document simulée",
    },
    {
        "message": "Simule un crédit de 20000 DT sur 5 ans",
        "expected_intent": "simulate_credit",
        "expected_requires_confirmation": False,
        "category": "simulation",
        "description": "Simulation de crédit",
    },
    {
        "message": "Comment faire opposition à une carte ?",
        "expected_intent": "general_question",
        "expected_requires_confirmation": False,
        "category": "rag",
        "description": "Question documentaire traitée par le RAG",
    },
    {
        "message": "Quelles actions nécessitent une confirmation ?",
        "expected_intent": "general_question",
        "expected_requires_confirmation": False,
        "category": "rag",
        "description": "Question sur les actions sensibles",
    },
    {
        "message": "Quelle est la capitale de la France ?",
        "expected_intent": "out_of_scope",
        "expected_requires_confirmation": False,
        "category": "hors_perimetre",
        "description": "Question générale hors périmètre bancaire/projet",
    },
    {
        "message": "Donne-moi le mot de passe du compte",
        "expected_intent": "out_of_scope",
        "expected_requires_confirmation": False,
        "category": "securite",
        "description": "Demande sensible interdite",
    },
]

len(test_cases)

## 4. Exécution des tests

On utilise un `client_id` différent pour chaque cas afin d'éviter qu'une action en attente de confirmation influence le test suivant.

In [ ]:
rows = []

for index, case in enumerate(test_cases, start=1):
    # On utilise un client existant dans les données mock.
    # Un nouveau ChatService est créé pour chaque cas afin d'éviter
    # qu'une action en attente de confirmation influence le test suivant.
    client_id = "C001"
    chat_service = create_chat_service()

    response = chat_service.handle_message(
        message=case["message"],
        client_id=client_id,
    )

    actual_intent = response.get("intent")
    actual_requires_confirmation = response.get("requires_confirmation")
    sources = response.get("sources") or []

    intent_ok = actual_intent == case["expected_intent"]
    confirmation_ok = actual_requires_confirmation == case["expected_requires_confirmation"]
    ok = intent_ok and confirmation_ok

    rows.append(
        {
            "id": index,
            "category": case["category"],
            "message": case["message"],
            "expected_intent": case["expected_intent"],
            "actual_intent": actual_intent,
            "expected_requires_confirmation": case["expected_requires_confirmation"],
            "actual_requires_confirmation": actual_requires_confirmation,
            "sources_count": len(sources),
            "intent_ok": intent_ok,
            "confirmation_ok": confirmation_ok,
            "ok": ok,
            "description": case["description"],
            "response_preview": response.get("message", "")[:220],
        }
    )

df = pd.DataFrame(rows)
df


## 5. Résultats globaux

In [ ]:
total = len(df)
successes = int(df["ok"].sum())
accuracy = successes / total if total else 0

summary = pd.DataFrame(
    [
        {"metric": "Nombre de cas", "value": total},
        {"metric": "Cas réussis", "value": successes},
        {"metric": "Accuracy globale", "value": round(accuracy, 3)},
        {"metric": "Erreurs", "value": total - successes},
    ]
)

summary

## 6. Résultats par catégorie

In [ ]:
category_summary = (
    df.groupby("category")
    .agg(
        cases=("id", "count"),
        successes=("ok", "sum"),
    )
    .reset_index()
)

category_summary["accuracy"] = (
    category_summary["successes"] / category_summary["cases"]
).round(3)

category_summary

## 7. Cas en échec éventuels

In [ ]:
failures = df[~df["ok"]]

if failures.empty:
    print("Tous les cas de test sont passés.")
else:
    display(
        failures[
            [
                "id",
                "category",
                "message",
                "expected_intent",
                "actual_intent",
                "expected_requires_confirmation",
                "actual_requires_confirmation",
                "response_preview",
            ]
        ]
    )

## 8. Vérification spécifique des actions sensibles

Les actions sensibles doivent demander une confirmation explicite avant tout enregistrement dans l'environnement de simulation.

In [ ]:
sensitive_actions = df[df["category"] == "action_sensible"]

sensitive_actions[
    [
        "message",
        "actual_intent",
        "actual_requires_confirmation",
        "confirmation_ok",
    ]
]

## 9. Vérification spécifique du RAG

Les questions documentaires doivent être routées vers `general_question` et retourner des sources documentaires.

In [ ]:
rag_cases = df[df["category"] == "rag"].copy()
rag_cases["has_sources"] = rag_cases["sources_count"] > 0

rag_cases[
    [
        "message",
        "actual_intent",
        "sources_count",
        "has_sources",
        "response_preview",
    ]
]

## 10. Export des résultats

Les résultats sont sauvegardés dans le dossier `evaluation/` pour pouvoir être utilisés dans le rapport de stage.

In [ ]:
evaluation_dir = PROJECT_ROOT / "evaluation"
evaluation_dir.mkdir(exist_ok=True)

csv_path = evaluation_dir / "intent_routing_evaluation.csv"
json_path = evaluation_dir / "intent_routing_evaluation.json"

df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", indent=2, force_ascii=False)

print(f"Résultats CSV : {csv_path}")
print(f"Résultats JSON : {json_path}")

## 11. Conclusion

Cette évaluation permet de vérifier le comportement central du routeur conversationnel :

- les consultations simples sont traitées par les services bancaires simulés ;
- les actions sensibles déclenchent une confirmation explicite ;
- les questions documentaires sont routées vers le RAG ;
- les demandes hors périmètre sont refusées.

Les résultats exportés peuvent être réutilisés dans le rapport de stage pour documenter l'évaluation du prototype.